In [1]:
!nvidia-smi


Tue Aug 18 03:51:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q "transformers==4.39.3" "tokenizers==0.15.2" "huggingface_hub==0.23.4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.6/402.6 kB 39.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.19.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.23.4 which is incompatible.
diffusers 0.39.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.23.4 which is incompatible.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.23.4 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.23.4 which is incompatible.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, but 

In [1]:
from google.colab import files

uploaded = files.upload()   # select splits.zip
!unzip -o splits.zip -d data
!ls -la data

Saving splits.zip to splits.zip
Archive:  splits.zip
  inflating: data/train.csv          
  inflating: data/val.csv            
  inflating: data/test_in.csv        
  inflating: data/test_idoos.csv     
  inflating: data/test_oodoos.csv    
  inflating: data/label_map.json     
total 1124
drwxr-xr-x 2 root root   4096 Aug 18 03:58 .
drwxr-xr-x 1 root root   4096 Aug 18 03:58 ..
-rw-r--r-- 1 root root   2141 Aug 17 11:33 label_map.json
-rw-r--r-- 1 root root  19443 Aug 17 11:33 test_idoos.csv
-rw-r--r-- 1 root root 226289 Aug 17 11:33 test_in.csv
-rw-r--r-- 1 root root  56144 Aug 17 11:33 test_oodoos.csv
-rw-r--r-- 1 root root 705094 Aug 17 11:33 train.csv
-rw-r--r-- 1 root root 121278 Aug 17 11:33 val.csv


In [9]:
import json, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 64
BATCH_SIZE = 32
EPOCHS = 10
LR = 3e-5
WARMUP_RATIO = 0.1
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| torch:", torch.__version__)

splits = {n: pd.read_csv(f"data/{n}.csv") for n in
          ["train", "val", "test_in", "test_idoos", "test_oodoos"]}
label_map = json.loads(open("data/label_map.json").read())
NUM_LABELS = len(label_map)

for k, v in splits.items():
    print(f"{k:12s} {v.shape}")
print("num_labels:", NUM_LABELS)

device: cuda | torch: 2.11.0+cu128
train        (8103, 3)
val          (1430, 3)
test_in      (2800, 3)
test_idoos   (280, 3)
test_oodoos  (1000, 3)
num_labels: 70


In [3]:
class QueryDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]


def make_collate(tokenizer):
    def collate(batch):
        texts, labels = zip(*batch)
        enc = tokenizer(
            list(texts),
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        enc["labels"] = torch.tensor(labels, dtype=torch.long)
        return enc
    return collate


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collate = make_collate(tokenizer)

train_loader = DataLoader(
    QueryDataset(splits["train"]["text"], splits["train"]["label"]),
    batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, drop_last=False)

val_loader = DataLoader(
    QueryDataset(splits["val"]["text"], splits["val"]["label"]),
    batch_size=64, shuffle=False, collate_fn=collate)

print("train batches:", len(train_loader), "| val batches:", len(val_loader))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

train batches: 254 | val batches: 23


In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_RATIO * total_steps),
    num_training_steps=total_steps,
)
scaler = torch.amp.GradScaler("cuda")

n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params/1e6:.1f}M | total steps: {total_steps}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


parameters: 67.0M | total steps: 2540


In [11]:

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, all_logits, all_labels = 0.0, [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.float16):
            out = model(**batch)
        total_loss += out.loss.item() * batch["labels"].size(0)
        all_logits.append(out.logits.float().cpu())
        all_labels.append(batch["labels"].cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    preds = logits.argmax(axis=1)
    return {
        "loss": total_loss / len(labels),
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }, logits


history, best_loss, best_epoch, best_state = [], float("inf"), None, None

for epoch in range(1, EPOCHS + 1):
    model.train()
    running, t0 = 0.0, time.time()

    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            out = model(**batch)

        scaler.scale(out.loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += out.loss.item()

    val_scores, _ = evaluate(val_loader)
    history.append({"epoch": epoch, "train_loss": running / len(train_loader), **val_scores})
    print(f"epoch {epoch:2d}: train_loss={running/len(train_loader):.4f} "
          f"val_loss={val_scores['loss']:.4f} val_macro_f1={val_scores['macro_f1']:.4f} "
          f"({time.time()-t0:.0f}s)")

    if val_scores["loss"] < best_loss:
        best_loss, best_epoch = val_scores["loss"], epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"   ↳ new best val_loss (epoch {epoch})")

model.load_state_dict(best_state)
print(f"\nrestored checkpoint from epoch {best_epoch} (val_loss={best_loss:.4f})")
pd.DataFrame(history)


epoch  1: train_loss=3.9299 val_loss=2.8839 val_macro_f1=0.5184 (17s)
   ↳ new best val_loss (epoch 1)
epoch  2: train_loss=1.9042 val_loss=1.0430 val_macro_f1=0.8067 (18s)
   ↳ new best val_loss (epoch 2)
epoch  3: train_loss=0.7304 val_loss=0.5275 val_macro_f1=0.8841 (17s)
   ↳ new best val_loss (epoch 3)
epoch  4: train_loss=0.3490 val_loss=0.3950 val_macro_f1=0.8993 (17s)
   ↳ new best val_loss (epoch 4)
epoch  5: train_loss=0.1981 val_loss=0.3425 val_macro_f1=0.9124 (17s)
   ↳ new best val_loss (epoch 5)
epoch  6: train_loss=0.1142 val_loss=0.3257 val_macro_f1=0.9188 (17s)
   ↳ new best val_loss (epoch 6)
epoch  7: train_loss=0.0709 val_loss=0.3383 val_macro_f1=0.9139 (17s)
epoch  8: train_loss=0.0475 val_loss=0.3445 val_macro_f1=0.9106 (17s)
epoch  9: train_loss=0.0334 val_loss=0.3402 val_macro_f1=0.9152 (17s)
epoch 10: train_loss=0.0272 val_loss=0.3394 val_macro_f1=0.9136 (17s)

restored checkpoint from epoch 6 (val_loss=0.3257)


,epoch,train_loss,loss,accuracy,macro_f1
0,1,3.929893,2.883875,0.573427,0.518354
1,2,1.904179,1.043007,0.820979,0.806701
2,3,0.730447,0.527467,0.886713,0.884146
3,4,0.348989,0.394997,0.897203,0.899251
4,5,0.198059,0.342492,0.910490,0.912357
5,6,0.114198,0.325742,0.917483,0.918785
6,7,0.070921,0.338280,0.912587,0.913866
7,8,0.047527,0.344521,0.908392,0.910574
8,9,0.033425,0.340233,0.913287,0.915214
9,10,0.027164,0.339355,0.911888,0.913646


In [12]:
@torch.no_grad()
def predict_logits(texts, batch_size=64):
    model.eval()
    chunks = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(
            texts[i:i + batch_size],
            padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors="pt",
        ).to(device)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = model(**enc).logits
        chunks.append(logits.float().cpu().numpy())
    return np.concatenate(chunks)


import os
os.makedirs("outputs", exist_ok=True)

for name in ["val", "test_in", "test_idoos", "test_oodoos"]:
    texts = splits[name]["text"].tolist()
    logits = predict_logits(texts)
    np.save(f"outputs/logits_{name}.npy", logits.astype(np.float32))
    print(f"{name:12s} logits {logits.shape}")

y_test = splits["test_in"]["label"].to_numpy()
test_preds = np.load("outputs/logits_test_in.npy").argmax(axis=1)
test_scores = {
    "accuracy": float(accuracy_score(y_test, test_preds)),
    "macro_f1": float(f1_score(y_test, test_preds, average="macro")),
}
print("\nTEST in-scope:", {k: round(v, 4) for k, v in test_scores.items()})

val          logits (1430, 70)
test_in      logits (2800, 70)
test_idoos   logits (280, 70)
test_oodoos  logits (1000, 70)

TEST in-scope: {'accuracy': 0.9207, 'macro_f1': 0.9209}


In [15]:
import json, shutil
from google.colab import files

model.save_pretrained("outputs/distilbert_intent")
tokenizer.save_pretrained("outputs/distilbert_intent")

json.dump({
    "model_name": MODEL_NAME,
    "num_labels": NUM_LABELS,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "epochs_run": EPOCHS,
    "learning_rate": LR,
    "warmup_ratio": WARMUP_RATIO,
    "selection": "lowest validation loss",
    "best_epoch": best_epoch,
    "best_val_loss": best_loss,
    "history": history,
    "test_in_scope": test_scores,
}, open("outputs/track_c_results.json", "w"), indent=2)

shutil.make_archive("track_c", "zip", "outputs")
print("archive size (MB):", round(os.path.getsize("track_c.zip") / 1e6, 1))
files.download("track_c.zip")

archive size (MB): 248.3


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>